# 案例 08 · 连接池泄漏打满 RDS，笔记接口报 1040

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

详情页异常分支未归还数据库连接，连接数缓慢上涨到 `max_connections=50`，应用报 `1040 Too many connections`，SLB 健康检查也随之失败。

## 练习目标

- 用 `SHOW PROCESSLIST` 区分“连接泄漏”与“慢查询”
- KILL 长空闲会话止血
- 修正连接池参数（pre_ping/recycle/timeout）
- 用 `rds_session_guard.py` 完成自动化处置

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-08-rds-conn")
lab.reset(start_api=True)
lab.start_mysql()
lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_rds_pool_leak(connections=60)
print("已注入：60 条应用连接（max_connections=50）")


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**会话列表**

In [ ]:
print(lab.sh("mysql -e SHOW PROCESSLIST"))

**连接统计**

In [ ]:
print(json.dumps(lab.mysql_stats(), ensure_ascii=False, indent=2))

**实例参数**

In [ ]:
print(lab.sh("mysql -e SHOW VARIABLES LIKE max_connections"))

**应用日志**

In [ ]:
print(lab.sh("tail -n 4 /var/log/xhs/note-api.log"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
st = lab.mysql_stats()
rows = [r for r in st.get("sessions", [])]
print("Threads_connected:", st.get("threads_connected"), "/ max:", st.get("max_connections"))
print("判断：连接数触顶且大量 Sleep → 连接泄漏；慢查询会表现为 Query 状态长时间占用")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
def sql(q):
    url = f"http://127.0.0.1:{lab.ports['mysql_ctrl']}/sql?q={urllib.parse.quote(q)}"
    with urllib.request.urlopen(url, timeout=5) as r:
        return json.loads(r.read()).get("rows", [])

rows = sql("show processlist")
killed = 0
for r in rows:
    if r["Command"] == "Sleep" and int(r["Time"]) >= 1:
        sql(f"kill {r['Id']}")
        killed += 1
print("已 KILL", killed, "个空闲会话")

lab.write("/etc/xhs/note-api.conf",
          "# 连接池已修复\nlisten_port = %d\n"
          "db.pool.max_open = 20\ndb.pool.max_idle = 5\n"
          "db.pool.idle_timeout = 30s\ndb.pool.health_check = true\n" % lab.ports["api"])
lab.state["faults"].pop("rds", None)
lab._save()
print("Threads_connected:", lab.mysql_stats().get("threads_connected"))


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('连接数回落', bool(lab.mysql_stats().get('threads_connected', 999) < 30))
lab.check('池参数已修正', bool('db.pool.health_check = true' in lab.path('/etc/xhs/note-api.conf').read_text()))

## 步骤 6 · 自动化沉淀

脚本统计 Sleep 会话与来源，KILL 长空闲连接，写入连接池修正参数，最后校验 `Threads_connected` 回落。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.start_mysql()
lab.inject_rds_pool_leak(connections=60)
out = lab.auto('rds_session_guard.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. `Sleep` 多与 `Query` 多分别指向什么问题？
2. 为什么 `pool_recycle` 必须小于 RDS 的 `wait_timeout`？
3. 泄漏来自无法修改的第三方 SDK 时，如何兜底？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')